In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")
POLICY_DIR = PROJECT_DIR / "data" / "policies"
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

assert PROJECT_DIR.exists(), "ShopAssist project folder not found."
assert POLICY_DIR.exists(), "Policy folder not found."

policy_files = sorted(POLICY_DIR.glob("*.md"))

print("Project:", PROJECT_DIR)
print("Policy documents:", len(policy_files))
print("Evaluation folder exists:", EVALUATION_DIR.exists())

Project: /content/drive/MyDrive/ShopAssist
Policy documents: 15
Evaluation folder exists: True


In [3]:
print("Policy files:")
for file in policy_files:
    print("-", file.name)

print("\nSaved evaluation files:")
for file in sorted(EVALUATION_DIR.glob("*.csv")):
    print("-", file.name)

Policy files:
- account_policy.md
- cancellation_policy.md
- damaged_items.md
- delivery_policy.md
- discount_policy.md
- gift_cards.md
- international_shipping.md
- missing_items.md
- payment_policy.md
- privacy_policy.md
- refund_policy.md
- replacement_policy.md
- returns_policy.md
- shipping_policy.md
- warranty_policy.md

Saved evaluation files:
- evaluation_questions.csv
- human_review_benchmark.csv
- improved_evaluation_report.csv
- rag_evaluation_results.csv


In [4]:
# Install the retrieval libraries if needed
%pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.3 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

# Load the existing policy documents
documents = []

for file_path in sorted(POLICY_DIR.glob("*.md")):
    documents.append({
        "source": file_path.name,
        "text": file_path.read_text(encoding="utf-8")
    })

documents_df = pd.DataFrame(documents)

# Load the same embedding model as the original notebook
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Create embeddings for the policy documents
document_embeddings = embedding_model.encode(
    documents_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Build the FAISS similarity-search index
index = faiss.IndexFlatIP(document_embeddings.shape[1])
index.add(document_embeddings.astype("float32"))

print("Retrieval system loaded successfully.")
print("Policy documents:", len(documents_df))
print("Embedding shape:", document_embeddings.shape)
print("FAISS index size:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval system loaded successfully.
Policy documents: 15
Embedding shape: (15, 384)
FAISS index size: 15


In [6]:
def search_policies(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    return pd.DataFrame([
        {
            "rank": rank,
            "source": documents_df.iloc[idx]["source"],
            "similarity": round(float(score), 4)
        }
        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0]), start=1
        )
        if idx >= 0
    ])


search_policies("What if my item arrived broken?")

,rank,source,similarity
0,1,damaged_items.md,0.6734
1,2,missing_items.md,0.5790
2,3,replacement_policy.md,0.5411


In [7]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Add GEMINI_API_KEY to Colab Secrets."

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client configured.")

Gemini client configured.


In [8]:

# Conversation memory
conversation_history = []


def add_to_memory(user_message, assistant_message):
    """Store a customer-assistant exchange."""
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })


def resolve_follow_up_query(query):
    """Add recent customer context to likely follow-up questions."""
    follow_up_starters = (
        "what about", "what if", "how about",
        "and if", "what happens if", "can they",
        "will they", "can i do that"
    )

    query_lower = query.lower().strip()

    if query_lower.startswith(follow_up_starters):
        previous_questions = [
            item["content"]
            for item in conversation_history[-6:]
            if item["role"] == "user"
        ]

        if previous_questions:
            return (
                "Previous customer context: "
                + " ".join(previous_questions[-2:])
                + ". Current follow-up question: "
                + query
            )

    return query


def chat_with_shopassist(user_message, top_k=3):
    """Generate a policy-grounded answer and store the exchange."""

    # Retrieve policies using the current question and relevant context
    resolved_query = resolve_follow_up_query(user_message)
    retrieved = search_policies(resolved_query, top_k=top_k)

    # Prepare policy context
    context_parts = []

    for _, row in retrieved.iterrows():
        policy_text = documents_df.loc[
            documents_df["source"] == row["source"], "text"
        ].iloc[0]

        context_parts.append(
            f"Source: {row['source']}\n{policy_text}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # Include recent conversation history
    history_text = "\n".join(
        f"{item['role'].capitalize()}: {item['content']}"
        for item in conversation_history[-6:]
    )

    prompt = f"""
You are ShopAssist AI, an e-commerce customer support assistant.

Rules:
- Answer using only the supplied policy information.
- Use conversation history to understand follow-up questions.
- Do not invent policies or promise outcomes.
- If the policies do not answer the question, say the information
  is unavailable.
- Be concise, helpful, and professional.
- Mention relevant policy filenames when appropriate.

CONVERSATION HISTORY:
{history_text if history_text else "No previous conversation."}

RELEVANT POLICIES:
{context}

CUSTOMER QUESTION:
{user_message}

ANSWER:
"""

    # One Gemini API request
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    answer = response.text or (
        "I'm sorry, I couldn't generate a response. Please try again."
    )

    # Save only after a response has been generated
    add_to_memory(user_message, answer)

    return {
        "answer": answer,
        "retrieved_documents": retrieved
    }


print("ShopAssist chatbot functions are ready.")


ShopAssist chatbot functions are ready.


In [9]:
def chat_with_shopassist(user_message, top_k=3):
    """Answer a customer using retrieved policies and recent chat history."""

    # Resolve the question using previous conversation
    resolved_query = resolve_follow_up_query(user_message)

    # Retrieve relevant policy documents
    retrieved = search_policies(resolved_query, top_k=top_k)

    # Build policy context
    context_parts = []

    for _, row in retrieved.iterrows():
        policy_text = documents_df.loc[
            documents_df["source"] == row["source"], "text"
        ].iloc[0]

        context_parts.append(
            f"Source: {row['source']}\n{policy_text}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # Include recent conversation
    history_text = "\n".join(
        f"{message['role'].capitalize()}: {message['content']}"
        for message in conversation_history[-6:]
    )

    prompt = f"""
You are ShopAssist AI, an e-commerce customer support assistant.

Rules:
- Answer using only the policy information provided.
- Use conversation history to understand follow-up questions.
- Never invent policy details or promise an outcome not stated in a policy.
- If the policies do not answer the question, say the information is unavailable.
- Give a concise, helpful response.
- Mention the relevant policy filename when appropriate.

CONVERSATION HISTORY:
{history_text if history_text else "No previous conversation."}

RELEVANT POLICIES:
{context}

CURRENT CUSTOMER QUESTION:
{user_message}

ANSWER:
"""

    # Call Gemini only when this function is explicitly run
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    answer = response.text or (
        "I'm sorry, I couldn't generate a response just now. Please try again."
    )

    # Store the original customer question and the generated answer
    add_to_memory(user_message, answer)

    return {
        "answer": answer,
        "retrieved_documents": retrieved
    }

In [10]:
result = chat_with_shopassist(
    "My item arrived damaged. What should I do?"
)

print("ShopAssist AI:")
print(result["answer"])

print("\nRetrieved policies:")
display(result["retrieved_documents"])

ShopAssist AI:
You should contact support as soon as possible. Please note that you may be asked to provide photographs of the damaged product, the packaging, and the shipping label, and you should not dispose of the damaged product or packaging until support has advised you to do so (Source: damaged_items.md). Additionally, replacement requests should normally be submitted within 30 days of delivery (Source: replacement_policy.md).

Retrieved policies:


,rank,source,similarity
0,1,damaged_items.md,0.6573
1,2,missing_items.md,0.4786
2,3,replacement_policy.md,0.4662


In [11]:
%pip install -q gradio

In [12]:
import gradio as gr


def respond(message, history):
    """Send a customer message to ShopAssist and return its answer."""

    if not message or not message.strip():
        return "Please enter a question."

    try:
        result = chat_with_shopassist(message.strip())
        return result["answer"]

    except Exception as error:
        error_text = str(error).lower()

        if "429" in error_text or "resource_exhausted" in error_text:
            return (
                "I'm temporarily unable to generate a response because "
                "the AI service has reached its usage limit. Please try "
                "again later."
            )

        print("Chatbot error:", repr(error))
        return (
            "Sorry, something went wrong while processing your question. "
            "Please try again later."
        )


demo = gr.ChatInterface(
    fn=respond,
    title="ShopAssist AI",
    description=(
        "Your e-commerce customer support assistant. "
        "Ask about delivery, returns, refunds, replacements, and more."
    ),
    examples=[
        "How long does standard delivery take?",
        "How can I return an item?",
        "My item arrived damaged. What should I do?",
        "What payment methods do you accept?"
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://61083d8a6d639a375b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://61083d8a6d639a375b.gradio.live


In [13]:
import gradio as gr


def respond(message, history):
    """Answer using this Gradio session's conversation history."""

    message = message.strip()
    if not message:
        return "Please enter a question."

    # Keep only recent messages from this user's current chat.
    recent_history = history[-6:] if history else []

    # Support Gradio message dictionaries and older tuple-style history.
    history_lines = []
    previous_user_messages = []

    for item in recent_history:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")
            if isinstance(content, str) and role in ("user", "assistant"):
                history_lines.append(f"{role.capitalize()}: {content}")
                if role == "user":
                    previous_user_messages.append(content)

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if user_text:
                history_lines.append(f"User: {user_text}")
                previous_user_messages.append(str(user_text))
            if assistant_text:
                history_lines.append(f"Assistant: {assistant_text}")

    # Use prior context for likely follow-up questions.
    follow_up_starters = (
        "what about", "what if", "how about", "and if",
        "can they", "will they", "what happens if",
        "what should i do", "can i do that"
    )

    is_follow_up = message.lower().startswith(follow_up_starters)

    if is_follow_up and previous_user_messages:
        retrieval_query = (
            "Previous customer context: "
            + " ".join(previous_user_messages[-2:])
            + ". Current question: "
            + message
        )
    else:
        retrieval_query = message

    retrieved = search_policies(retrieval_query, top_k=3)

    policy_context = []

    for _, row in retrieved.iterrows():
        policy_text = documents_df.loc[
            documents_df["source"] == row["source"], "text"
        ].iloc[0]

        policy_context.append(
            f"Source: {row['source']}\n{policy_text}"
        )

    prompt = f"""
You are ShopAssist AI, an e-commerce customer support assistant.

Answer using only the policy information below.
Use the conversation history to understand follow-up questions.
Do not invent policy rules or promise outcomes.
If the policies do not answer the question, say so clearly.
Give a concise, helpful answer and cite the relevant policy filename.

CONVERSATION HISTORY:
{chr(10).join(history_lines) if history_lines else "No previous conversation."}

RELEVANT POLICIES:
{"".join(policy_context)}

CURRENT CUSTOMER QUESTION:
{message}

ANSWER:
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )

        answer = response.text

        if not answer:
            return "I couldn't generate an answer just now. Please try again later."

        return answer

    except Exception as error:
        error_text = str(error).lower()

        if "429" in error_text or "resource_exhausted" in error_text:
            return (
                "The AI service has reached its current usage limit. "
                "Please try again later."
            )

        print("Chatbot error:", repr(error))
        return "Sorry, something went wrong. Please try again later."


demo = gr.ChatInterface(
    fn=respond,
    title="ShopAssist AI",
    description=(
        "Your e-commerce customer support assistant. "
        "Ask about delivery, returns, refunds, and replacements."
    ),
    examples=[
        "How long does standard delivery take?",
        "How can I return an item?",
        "My item arrived damaged. What should I do?"
    ]
)

print("Session-aware chatbot interface is ready.")

Session-aware chatbot interface is ready.


In [14]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74a5a099b1fbf04834.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://74a5a099b1fbf04834.gradio.live


In [15]:

for folder in ["src", "app", "tests", "data/policies", "data/evaluation"]:
    path = PROJECT_DIR / folder
    print(f"\n{folder}/")

    if path.exists():
        for item in sorted(path.iterdir()):
            print("  ", item.name)
    else:
        print("   Folder does not exist")



src/
   __pycache__
   chatbot.py
   retrieval.py

app/
   gradio_app.py

tests/
   test_retrieval.py

data/policies/
   account_policy.md
   cancellation_policy.md
   damaged_items.md
   delivery_policy.md
   discount_policy.md
   gift_cards.md
   international_shipping.md
   missing_items.md
   payment_policy.md
   privacy_policy.md
   refund_policy.md
   replacement_policy.md
   returns_policy.md
   shipping_policy.md
   warranty_policy.md

data/evaluation/
   evaluation_questions.csv
   evaluation_questions.gsheet
   human_review_benchmark.csv
   human_review_benchmark.gsheet
   improved_evaluation_report.csv
   rag_evaluation_results.csv


In [16]:
from pathlib import Path

retrieval_code = r'''
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


def load_policies(policy_dir):
    """Load all Markdown policy files from a directory."""
    policy_dir = Path(policy_dir)
    documents = []

    for file_path in sorted(policy_dir.glob("*.md")):
        documents.append({
            "source": file_path.name,
            "text": file_path.read_text(encoding="utf-8")
        })

    if not documents:
        raise ValueError(f"No Markdown policy files found in {policy_dir}")

    return pd.DataFrame(documents)


def build_retrieval_index(documents_df, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    """Create normalized embeddings and a FAISS similarity-search index."""
    model = SentenceTransformer(model_name)

    embeddings = model.encode(
        documents_df["text"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return model, index


def search_policies(query, documents_df, model, index, top_k=3):
    """Return the top matching policy documents and similarity scores."""
    if not query or not query.strip():
        return pd.DataFrame(columns=["rank", "source", "similarity"])

    query_embedding = model.encode(
        [query.strip()],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, len(documents_df))
    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        if idx >= 0:
            results.append({
                "rank": rank,
                "source": documents_df.iloc[idx]["source"],
                "similarity": float(score)
            })

    return pd.DataFrame(results)
'''

project_src = PROJECT_DIR / "src"
project_src.mkdir(parents=True, exist_ok=True)

retrieval_file = project_src / "retrieval.py"
retrieval_file.write_text(retrieval_code.strip() + "\n", encoding="utf-8")

print("Created:", retrieval_file)
print("File size:", retrieval_file.stat().st_size, "bytes")

Created: /content/drive/MyDrive/ShopAssist/src/retrieval.py
File size: 1949 bytes


In [17]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.retrieval import (
    load_policies,
    build_retrieval_index,
    search_policies
)

test_documents = load_policies(PROJECT_DIR / "data" / "policies")
test_model, test_index = build_retrieval_index(test_documents)

test_results = search_policies(
    "My item arrived damaged. What should I do?",
    test_documents,
    test_model,
    test_index,
    top_k=3
)

print("Policies loaded:", len(test_documents))
display(test_results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Policies loaded: 15


,rank,source,similarity
0,1,damaged_items.md,0.657310
1,2,missing_items.md,0.478592
2,3,replacement_policy.md,0.466196


In [18]:
chatbot_code = r'''
import os

from google import genai

from src.retrieval import search_policies


def create_chatbot(
    documents_df,
    embedding_model,
    faiss_index,
    api_key=None,
    model_name="gemini-3.5-flash-lite"
):
    """Create a policy-grounded ShopAssist chatbot."""

    api_key = api_key or os.environ.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(
            "Gemini API key not found. Pass api_key explicitly."
        )

    client = genai.Client(api_key=api_key)

    def answer_question(question, history=None):
        if not question or not question.strip():
            return "Please enter a question."

        history = history or []

        # Retrieve relevant policies
        results = search_policies(
            question,
            documents_df,
            embedding_model,
            faiss_index,
            top_k=3
        )

        policy_context = []

        for _, row in results.iterrows():
            policy_text = documents_df.loc[
                documents_df["source"] == row["source"],
                "text"
            ].iloc[0]

            policy_context.append(
                f"Source: {row['source']}\n{policy_text}"
            )

        context_text = "\n\n".join(policy_context)

        # Keep only recent conversation turns
        history_text = ""
        for item in history[-6:]:
            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")
                if role in ("user", "assistant") and isinstance(content, str):
                    history_text += f"{role.capitalize()}: {content}\n"
            elif isinstance(item, (list, tuple)) and len(item) == 2:
                user_text, assistant_text = item
                if user_text:
                    history_text += f"User: {user_text}\n"
                if assistant_text:
                    history_text += f"Assistant: {assistant_text}\n"

        prompt = f"""
You are ShopAssist AI, an e-commerce customer support assistant.

Answer the customer's question using only the supplied policy documents.
Use the conversation history to understand follow-up questions.

Rules:
1. Do not invent policies, deadlines, exceptions, or guarantees.
2. If the policies do not answer the question, clearly say that the
   available policies do not specify the answer.
3. Give clear, concise, helpful next steps when supported by the policies.
4. Cite the relevant policy filenames in your answer, for example:
   (Source: returns_policy.md).
5. Do not claim an action has been completed or that you contacted support.
6. Never ask the customer to provide passwords or full card numbers.

Relevant policy documents:
{context_text}

Recent conversation:
{history_text if history_text else "No previous conversation."}

Current customer question:
{question.strip()}
"""

        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            answer = response.text

            if not answer:
                return "I couldn't generate an answer. Please try again."

            return answer

        except Exception as error:
            error_text = str(error).lower()

            if "429" in error_text or "resource_exhausted" in error_text:
                return (
                    "The AI service has reached its current usage limit. "
                    "Please try again later."
                )

            return (
                "Sorry, I couldn't process your question right now. "
                "Please try again later."
            )

    return answer_question
'''

chatbot_file = PROJECT_DIR / "src" / "chatbot.py"
chatbot_file.write_text(chatbot_code.strip() + "\n", encoding="utf-8")

print("Created:", chatbot_file)
print("File size:", chatbot_file.stat().st_size, "bytes")

Created: /content/drive/MyDrive/ShopAssist/src/chatbot.py
File size: 3684 bytes


In [19]:
from google.colab import userdata
from src.chatbot import create_chatbot

# Retrieve the API key securely from Colab Secrets
gemini_key = userdata.get("GEMINI_API_KEY")

# Initialise the chatbot using the retrieval objects from Step 2
shopassist = create_chatbot(
    documents_df=test_documents,
    embedding_model=test_model,
    faiss_index=test_index,
    api_key=gemini_key
)

print("ShopAssist chatbot initialised successfully!")
print("Policy documents available:", len(test_documents))

ShopAssist chatbot initialised successfully!
Policy documents available: 15


In [20]:
app_dir = PROJECT_DIR / "app"
app_dir.mkdir(parents=True, exist_ok=True)

print("App folder:", app_dir)
print("Chatbot module exists:", (PROJECT_DIR / "src" / "chatbot.py").exists())
print("Retrieval module exists:", (PROJECT_DIR / "src" / "retrieval.py").exists())

App folder: /content/drive/MyDrive/ShopAssist/app
Chatbot module exists: True
Retrieval module exists: True


In [21]:
app_code = r'''
from pathlib import Path
import sys

import gradio as gr
from google.colab import userdata

# Locate the project root
PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.retrieval import (
    load_policies,
    build_retrieval_index
)
from src.chatbot import create_chatbot


def build_app():
    """Build the ShopAssist customer-support chat interface."""

    policy_dir = PROJECT_DIR / "data" / "policies"
    documents_df = load_policies(policy_dir)

    embedding_model, faiss_index = build_retrieval_index(documents_df)

    api_key = userdata.get("GEMINI_API_KEY")

    chatbot = create_chatbot(
        documents_df=documents_df,
        embedding_model=embedding_model,
        faiss_index=faiss_index,
        api_key=api_key
    )

    def respond(message, history):
        if not message or not message.strip():
            return "Please enter a question."

        try:
            return chatbot(message.strip(), history=history)
        except Exception:
            return (
                "Sorry, something went wrong. "
                "Please try asking your question again."
            )

    demo = gr.ChatInterface(
        fn=respond,
        title="ShopAssist AI",
        description=(
            "Your e-commerce support assistant. "
            "Ask about delivery, returns, refunds, replacements, and more. "
            "Answers are grounded in the available policy documents."
        ),
        examples=[
            "How long does standard delivery take?",
            "How can I return an item?",
            "My item arrived damaged. What should I do?",
            "Can I cancel my order after dispatch?"
        ]
    )

    return demo


if __name__ == "__main__":
    app = build_app()
    app.launch(share=True)
'''

app_dir = PROJECT_DIR / "app"
app_dir.mkdir(parents=True, exist_ok=True)

app_file = app_dir / "gradio_app.py"
app_file.write_text(app_code.strip() + "\n", encoding="utf-8")

print("Created:", app_file)
print("File size:", app_file.stat().st_size, "bytes")

Created: /content/drive/MyDrive/ShopAssist/app/gradio_app.py
File size: 1851 bytes


In [22]:
app_file = PROJECT_DIR / "app" / "gradio_app.py"

print("Application file exists:", app_file.exists())
print("File size:", app_file.stat().st_size, "bytes")
print("\nFirst 15 lines:\n")

print("\n".join(app_file.read_text(encoding="utf-8").splitlines()[:15]))

Application file exists: True
File size: 1851 bytes

First 15 lines:

from pathlib import Path
import sys

import gradio as gr
from google.colab import userdata

# Locate the project root
PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.retrieval import (
    load_policies,
    build_retrieval_index


In [23]:
requirements = """sentence-transformers
faiss-cpu
numpy
pandas
google-genai
gradio
"""

requirements_file = PROJECT_DIR / "requirements.txt"
requirements_file.write_text(requirements, encoding="utf-8")

print("Created:", requirements_file)
print("\nDependencies:")
print(requirements_file.read_text(encoding="utf-8"))

Created: /content/drive/MyDrive/ShopAssist/requirements.txt

Dependencies:
sentence-transformers
faiss-cpu
numpy
pandas
google-genai
gradio



In [24]:
for folder in [
    "src",
    "app",
    "tests",
    "data/policies",
    "data/evaluation"
]:
    path = PROJECT_DIR / folder
    print(f"\n{folder}/")

    for item in sorted(path.iterdir()):
        if item.is_file():
            print("  ", item.name)

print("\nRoot-level files:")
for item in sorted(PROJECT_DIR.iterdir()):
    if item.is_file():
        print("  ", item.name)


src/
   chatbot.py
   retrieval.py

app/
   gradio_app.py

tests/
   test_retrieval.py

data/policies/
   account_policy.md
   cancellation_policy.md
   damaged_items.md
   delivery_policy.md
   discount_policy.md
   gift_cards.md
   international_shipping.md
   missing_items.md
   payment_policy.md
   privacy_policy.md
   refund_policy.md
   replacement_policy.md
   returns_policy.md
   shipping_policy.md
   warranty_policy.md

data/evaluation/
   evaluation_questions.csv
   evaluation_questions.gsheet
   human_review_benchmark.csv
   human_review_benchmark.gsheet
   improved_evaluation_report.csv
   rag_evaluation_results.csv

Root-level files:
   .gitignore
   requirements.txt


In [25]:
test_code = r'''
import sys
import unittest
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.retrieval import (
    load_policies,
    build_retrieval_index,
    search_policies
)


class TestShopAssistRetrieval(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.documents_df = load_policies(
            PROJECT_DIR / "data" / "policies"
        )
        cls.model, cls.index = build_retrieval_index(cls.documents_df)

    def test_all_policy_documents_load(self):
        self.assertEqual(len(self.documents_df), 15)

    def test_policy_sources_are_markdown_files(self):
        self.assertTrue(
            all(source.endswith(".md")
                for source in self.documents_df["source"])
        )

    def test_damaged_item_query(self):
        results = search_policies(
            "My item arrived damaged. What should I do?",
            self.documents_df,
            self.model,
            self.index,
            top_k=3
        )
        self.assertFalse(results.empty)
        self.assertEqual(results.iloc[0]["source"], "damaged_items.md")

    def test_delivery_query(self):
        results = search_policies(
            "How long does standard delivery take?",
            self.documents_df,
            self.model,
            self.index,
            top_k=3
        )
        self.assertFalse(results.empty)
        self.assertEqual(results.iloc[0]["source"], "delivery_policy.md")

    def test_empty_query(self):
        results = search_policies(
            "",
            self.documents_df,
            self.model,
            self.index
        )
        self.assertTrue(results.empty)


if __name__ == "__main__":
    unittest.main(argv=["first-arg-is-ignored"], exit=False)
'''

tests_dir = PROJECT_DIR / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

test_file = tests_dir / "test_retrieval.py"
test_file.write_text(test_code.strip() + "\n", encoding="utf-8")

print("Created:", test_file)
print("File size:", test_file.stat().st_size, "bytes")

Created: /content/drive/MyDrive/ShopAssist/tests/test_retrieval.py
File size: 1839 bytes


In [26]:
%run /content/drive/MyDrive/ShopAssist/tests/test_retrieval.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

.....
----------------------------------------------------------------------
Ran 5 tests in 3.241s

OK


In [27]:
import sys
import unittest
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.chatbot import create_chatbot


class TestShopAssistChatbot(unittest.TestCase):

    def test_missing_api_key_raises_error(self):
        with self.assertRaisesRegex(ValueError, "Gemini API key not found"):
            create_chatbot(
                documents_df=None,
                embedding_model=None,
                faiss_index=None,
                api_key=""
            )


suite = unittest.TestLoader().loadTestsFromTestCase(
    TestShopAssistChatbot
)
unittest.TextTestRunner(verbosity=2).run(suite)

test_missing_api_key_raises_error (__main__.TestShopAssistChatbot.test_missing_api_key_raises_error) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.008s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [28]:
gitignore_content = """# Secrets and environment variables
.env
.env.*
!.env.example
*api_key*
*secret*

# Python cache and compiled files
__pycache__/
*.py[cod]
.pytest_cache/

# Virtual environments
.venv/
venv/
env/

# Notebook checkpoints
.ipynb_checkpoints/

# Model and temporary files
*.pkl
*.joblib
*.pt
*.pth
*.bin
*.safetensors
.cache/
tmp/
temp/
"""

gitignore_file = PROJECT_DIR / ".gitignore"
gitignore_file.write_text(gitignore_content, encoding="utf-8")

print("Created:", gitignore_file)
print(gitignore_file.read_text(encoding="utf-8"))

Created: /content/drive/MyDrive/ShopAssist/.gitignore
# Secrets and environment variables
.env
.env.*
!.env.example
*api_key*
*secret*

# Python cache and compiled files
__pycache__/
*.py[cod]
.pytest_cache/

# Virtual environments
.venv/
venv/
env/

# Notebook checkpoints
.ipynb_checkpoints/

# Model and temporary files
*.pkl
*.joblib
*.pt
*.pth
*.bin
*.safetensors
.cache/
tmp/
temp/



In [29]:
print("ShopAssist root directory:\n")

for item in sorted(PROJECT_DIR.iterdir()):
    suffix = "/" if item.is_dir() else ""
    print(f"  {item.name}{suffix}")

ShopAssist root directory:

  .gitignore
  app/
  data/
  notebooks/
  requirements.txt
  src/
  tests/


In [30]:
from google.colab import userdata
import subprocess
from pathlib import Path
import os

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets. "
        "Check the secret name without printing the token."
    )

repo_url = (
    "https://x-access-token:"
    + GITHUB_TOKEN
    + "@github.com/DhruviPatadia/ShopAssist.git"
)

clone_path = Path("/content/ShopAssist")

if (clone_path / ".git").exists():
    print("Git repository already exists:", clone_path)
else:
    result = subprocess.run(
        [
            "git", "clone",
            "https://github.com/DhruviPatadia/ShopAssist.git",
            str(clone_path)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("Clone failed:")
        print(result.stderr)
    else:
        print("Repository cloned successfully!")

print("\nRepository contents:")
if (clone_path / ".git").exists():
    for item in sorted(clone_path.iterdir()):
        print(" ", item.name)

Repository cloned successfully!

Repository contents:
  .git
  .gitignore
  README.md
  app
  data
  notebooks
  requirements.txt
  src
  tests


In [31]:
from pathlib import Path
import shutil

DRIVE_PROJECT = Path("/content/drive/MyDrive/ShopAssist")
GITHUB_PROJECT = Path("/content/ShopAssist")

# Copy only these new files/folders from Drive.
files_to_copy = [
    ".gitignore",
    "requirements.txt",
    "src",
    "app",
    "tests"
]

for name in files_to_copy:
    source = DRIVE_PROJECT / name
    destination = GITHUB_PROJECT / name

    if not source.exists():
        print(f"Skipped (not found): {name}")
        continue

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

    print(f"Copied: {name}")

print("\nExisting README and notebooks were not copied or overwritten.")

Copied: .gitignore
Copied: requirements.txt
Copied: src
Copied: app
Copied: tests

Existing README and notebooks were not copied or overwritten.


In [32]:
import subprocess

repo_path = "/content/ShopAssist"

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No changes detected.")

 M .gitignore
 M app/gradio_app.py



In [33]:
from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/ShopAssist")
GITHUB_PROJECT = Path("/content/ShopAssist")

files_to_check = [
    ".gitignore",
    "requirements.txt",
    "src/retrieval.py",
    "src/chatbot.py",
    "app/gradio_app.py",
    "tests/test_retrieval.py"
]

all_match = True

for name in files_to_check:
    drive_file = DRIVE_PROJECT / name
    github_file = GITHUB_PROJECT / name

    if not drive_file.exists() or not github_file.exists():
        print(f"MISSING: {name}")
        all_match = False
    elif drive_file.read_bytes() == github_file.read_bytes():
        print(f"MATCH:   {name}")
    else:
        print(f"DIFFER:  {name}")
        all_match = False

print("\nAll checked files match:", all_match)

MATCH:   .gitignore
MATCH:   requirements.txt
MATCH:   src/retrieval.py
MATCH:   src/chatbot.py
MATCH:   app/gradio_app.py
MATCH:   tests/test_retrieval.py

All checked files match: True


In [34]:
import subprocess
from google.colab import userdata

repo_path = "/content/ShopAssist"
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

def run_git(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=repo_path,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

# Configure the commit identity for this repository
run_git(["config", "user.name", "ShopAssist Project"])
run_git(["config", "user.email", "shopassist-project@users.noreply.github.com"])

# Stage only the intended additions
run_git([
    "add", "--",
    ".gitignore",
    "requirements.txt",
    "src/",
    "app/",
    "tests/"
])

# Create the commit
run_git([
    "commit", "-m",
    "Add modular ShopAssist chatbot application and tests"
])

print("Commit created successfully.")

# Use the token securely for the push without printing it
remote_url = (
    "https://x-access-token:"
    + token
    + "@github.com/DhruviPatadia/ShopAssist.git"
)

push_result = subprocess.run(
    ["git", "push", remote_url, "HEAD"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

if push_result.returncode == 0:
    print("Push successful!")
else:
    print("Push did not complete:")
    print(push_result.stderr)

Commit created successfully.
Push successful!


In [35]:
%cd /content/ShopAssist

!pip install -q -r requirements.txt

/content/ShopAssist


In [36]:
from pathlib import Path

app_file = Path("/content/ShopAssist/app/gradio_app.py")

assert app_file.exists(), "Gradio application file not found."

print("Application file found.")
print("Ready to launch ShopAssist.")

Application file found.
Ready to launch ShopAssist.


In [37]:
%cd /content/ShopAssist
!python app/gradio_app.py

/content/ShopAssist
Loading weights: 100% 103/103 [00:00<00:00, 10197.17it/s]
Batches: 100% 1/1 [00:01<00:00,  1.16s/it]
Traceback (most recent call last):
  File "/content/ShopAssist/app/gradio_app.py", line 69, in <module>
    app = build_app()
  File "/content/ShopAssist/app/gradio_app.py", line 28, in build_app
    api_key = userdata.get("GEMINI_API_KEY")
  File "/usr/local/lib/python3.13/dist-packages/google/colab/userdata.py", line 62, in get
    resp = _message.blocking_request(
        'GetSecret', request={'key': key}, timeout_sec=10
    )
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_message.py", line 173, in blocking_request
    request_id = send_request(
        request_type, request, parent=parent, expect_reply=True
    )
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_message.py", line 117, in send_request
    instance = ipython.get_kernelapp()
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_ipython.py", line 28, in get_kernelap

In [38]:
from pathlib import Path

app_file = Path("/content/ShopAssist/app/gradio_app.py")
app_code = app_file.read_text(encoding="utf-8")

app_code = app_code.replace(
    "from google.colab import userdata\n",
    "import os\n"
)

app_code = app_code.replace(
    '    api_key = userdata.get("GEMINI_API_KEY")',
    '    api_key = os.environ.get("GEMINI_API_KEY")'
)

app_file.write_text(app_code, encoding="utf-8")

print("Updated app/gradio_app.py")
print("Colab userdata dependency removed:", "from google.colab import userdata" not in app_code)
print("Environment variable lookup added:", 'os.environ.get("GEMINI_API_KEY")' in app_code)

Updated app/gradio_app.py
Colab userdata dependency removed: True
Environment variable lookup added: True


In [39]:
from google.colab import userdata
import os
import subprocess

# Retrieve the secret in the notebook's supported environment
gemini_key = userdata.get("GEMINI_API_KEY")

if not gemini_key:
    raise ValueError("GEMINI_API_KEY is missing from Colab Secrets.")

# Pass it to the child process without printing it
app_env = os.environ.copy()
app_env["GEMINI_API_KEY"] = gemini_key

process = subprocess.Popen(
    ["python", "/content/ShopAssist/app/gradio_app.py"],
    cwd="/content/ShopAssist",
    env=app_env
)

print("ShopAssist application process started.")
print("Process ID:", process.pid)

ShopAssist application process started.
Process ID: 10001


In [40]:
import time

time.sleep(3)

print("Process running:", process.poll() is None)

if process.poll() is None:
    print("ShopAssist is still running.")
    print("Check the next cell for the Gradio URL.")
else:
    print("Process exited with code:", process.returncode)
    print("Check the error output from the process.")

Process running: True
ShopAssist is still running.
Check the next cell for the Gradio URL.


In [41]:
process.terminate()
process.wait(timeout=10)

print("Background process stopped.")

Background process stopped.


In [42]:
from pathlib import Path

app_file = Path("/content/ShopAssist/app/gradio_app.py")
app_code = app_file.read_text(encoding="utf-8")

for line_number, line in enumerate(app_code.splitlines(), start=1):
    if "api_key" in line.lower() or "userdata" in line.lower():
        print(f"{line_number}: {line}")

28:     api_key = os.environ.get("GEMINI_API_KEY")
34:         api_key=api_key


In [43]:
import os
import subprocess
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = key

check = subprocess.run(
    [
        "python", "-c",
        "import os; print('Child process sees key:', bool(os.environ.get('GEMINI_API_KEY')))"
    ],
    env=env,
    capture_output=True,
    text=True
)

print(check.stdout)
print(check.stderr if check.returncode != 0 else "")

Child process sees key: True




In [44]:
from pathlib import Path

app_file = Path("/content/ShopAssist/app/gradio_app.py")
app_code = app_file.read_text(encoding="utf-8")

old_line = '    api_key = os.environ.get("GEMINI_API_KEY")'
new_line = '    api_key = os.environ.get("GEMINI_API_KEY", "").strip()'

if old_line in app_code:
    app_code = app_code.replace(old_line, new_line)
    app_file.write_text(app_code, encoding="utf-8")
    print("Updated API key lookup.")
elif new_line in app_code:
    print("API key lookup already updated.")
else:
    print("Expected API key line not found. Inspect the file before changing it.")

Updated API key lookup.


In [45]:
app_code = app_file.read_text(encoding="utf-8")

for number, line in enumerate(app_code.splitlines(), start=1):
    if "api_key" in line.lower():
        print(f"{number}: {line}")

28:     api_key = os.environ.get("GEMINI_API_KEY", "").strip()
34:         api_key=api_key


In [46]:
import gradio
print("Gradio version:", gradio.__version__)

Gradio version: 6.30.0


In [47]:
from pathlib import Path

app_file = Path("/content/ShopAssist/app/gradio_app.py")
lines = app_file.read_text(encoding="utf-8").splitlines()

for number, line in enumerate(lines, start=1):
    if "launch(" in line or "api_key =" in line or "build_app()" in line:
        print(f"{number}: {line}")

20: def build_app():
28:     api_key = os.environ.get("GEMINI_API_KEY", "").strip()
69:     app = build_app()
70:     app.launch(share=True)


In [48]:
import gradio as gr

demo = gr.Interface(
    fn=lambda text: f"ShopAssist test received: {text}",
    inputs=gr.Textbox(label="Test message"),
    outputs=gr.Textbox(label="Response"),
    title="ShopAssist Startup Test"
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3dc7047c1256f4e3c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [49]:
import os
import subprocess
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = key

result = subprocess.run(
    [
        "python", "-c",
        "import os; print('Gemini key available to app:', bool(os.environ.get('GEMINI_API_KEY')))"
    ],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Gemini key available to app: True




In [50]:
import os
import subprocess
from google.colab import userdata

gemini_key = userdata.get("GEMINI_API_KEY")

if not gemini_key:
    raise ValueError("GEMINI_API_KEY is unavailable.")

app_env = os.environ.copy()
app_env["GEMINI_API_KEY"] = gemini_key

print("Starting ShopAssist...")

result = subprocess.run(
    ["python", "-u", "app/gradio_app.py"],
    cwd="/content/ShopAssist",
    env=app_env,
    text=True,
    capture_output=True,
    timeout=120
)

print("Return code:", result.returncode)
print("\n--- Application output ---")
print(result.stdout)
print("\n--- Errors, if any ---")
print(result.stderr)

Starting ShopAssist...


TimeoutExpired: Command '['python', '-u', 'app/gradio_app.py']' timed out after 120 seconds

In [51]:
import os
import subprocess
import time
from pathlib import Path
from google.colab import userdata

gemini_key = userdata.get("GEMINI_API_KEY")
if not gemini_key:
    raise ValueError("GEMINI_API_KEY is unavailable.")

app_env = os.environ.copy()
app_env["GEMINI_API_KEY"] = gemini_key

log_file = Path("/content/shopassist_app.log")
log_handle = open(log_file, "w")

app_process = subprocess.Popen(
    ["python", "-u", "app/gradio_app.py"],
    cwd="/content/ShopAssist",
    env=app_env,
    stdout=log_handle,
    stderr=subprocess.STDOUT
)

print("Started process:", app_process.pid)
time.sleep(15)

print("\n--- Startup log ---")
print(log_file.read_text(encoding="utf-8", errors="replace"))

print("\nProcess running:", app_process.poll() is None)

Started process: 10885

--- Startup log ---


Process running: True


In [52]:
import time
from pathlib import Path

time.sleep(15)

log_file = Path("/content/shopassist_app.log")

print("Process running:", app_process.poll() is None)
print("Log size:", log_file.stat().st_size, "bytes")
print("\n--- Latest log output ---")
print(log_file.read_text(encoding="utf-8", errors="replace"))

Process running: True
Log size: 843 bytes

--- Latest log output ---

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4960.82it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://a8b0e52c265e38f9c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



In [53]:
from pathlib import Path
import subprocess

repo_path = Path("/content/ShopAssist")

print("Current Git status:")
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_path,
    capture_output=True,
    text=True
)
print(result.stdout or "No changes detected.")

Current Git status:
 M app/gradio_app.py
?? .gradio/



In [54]:
from pathlib import Path

repo_path = Path("/content/ShopAssist")
gradio_dir = repo_path / ".gradio"

if gradio_dir.exists():
    print("Files in .gradio/:")
    for item in sorted(gradio_dir.rglob("*")):
        if item.is_file():
            print(item.relative_to(repo_path))
else:
    print(".gradio/ does not exist.")

Files in .gradio/:
.gradio/certificate.pem


In [55]:
from pathlib import Path

repo_path = Path("/content/ShopAssist")
gitignore_file = repo_path / ".gitignore"

existing = gitignore_file.read_text(encoding="utf-8") if gitignore_file.exists() else ""

addition = "\n# Local Gradio configuration\n.gradio/\n"

if ".gradio/" not in existing:
    gitignore_file.write_text(existing.rstrip() + "\n" + addition, encoding="utf-8")

print("Updated .gitignore to ignore .gradio/")

Updated .gitignore to ignore .gradio/


In [56]:
import subprocess

result = subprocess.run(
    ["git", "status", "--short"],
    cwd="/content/ShopAssist",
    capture_output=True,
    text=True
)

print(result.stdout or "No changes detected.")

 M .gitignore
 M app/gradio_app.py



In [57]:
import subprocess
from google.colab import userdata

repo_path = "/content/ShopAssist"
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

def run_git(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=repo_path,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

# Stage only the two intended files
run_git(["add", "--", ".gitignore", "app/gradio_app.py"])

# Commit
run_git([
    "commit", "-m",
    "Fix app configuration and ignore local Gradio files"
])

print("Commit created successfully.")

# Push without displaying the token
remote_url = (
    "https://x-access-token:"
    + token
    + "@github.com/DhruviPatadia/ShopAssist.git"
)

result = subprocess.run(
    ["git", "push", remote_url, "HEAD"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Push successful!")
else:
    print("Push failed:")
    print(result.stderr)

Commit created successfully.
Push successful!


In [58]:
from pathlib import Path

readme_path = Path("/content/ShopAssist/README.md")

print("README exists:", readme_path.exists())

if readme_path.exists():
    print("\n--- Current README ---\n")
    print(readme_path.read_text(encoding="utf-8"))

README exists: True

--- Current README ---

# ShopAssist AI — Intelligent E-commerce Customer Support Agent

ShopAssist AI is a retrieval-augmented customer support chatbot designed to answer common e-commerce questions using a collection of policy documents. It combines semantic search, vector retrieval, a large language model, and a Gradio chat interface to provide answers grounded in relevant policies.

## Project Overview

E-commerce customers often need quick, consistent answers about delivery, returns, refunds, replacements, payments, and other order-related questions.

ShopAssist retrieves relevant policy documents and uses them as context for generating responses. It is designed to cite its policy sources and acknowledge when the available documentation does not specify an answer.

## Key Features

- **Semantic search:** Uses Sentence Transformers to encode policy documents and customer queries.
- **Vector retrieval:** Uses FAISS to retrieve relevant policies by similarity.
- 

In [59]:
from pathlib import Path

repo_path = Path("/content/ShopAssist")
readme_path = repo_path / "README.md"

readme_content = r"""
# ShopAssist AI — Intelligent E-commerce Customer Support Agent

ShopAssist AI is a retrieval-augmented customer support chatbot designed to answer common e-commerce questions using a collection of policy documents. It combines semantic search, vector retrieval, a large language model, and a Gradio chat interface to provide answers grounded in relevant policies.

## Project Overview

E-commerce customers often need quick, consistent answers about delivery, returns, refunds, replacements, payments, and other order-related questions.

ShopAssist retrieves relevant policy documents and uses them as context for generating responses. It is designed to cite its policy sources and acknowledge when the available documentation does not specify an answer.

## Key Features

- **Semantic search:** Uses Sentence Transformers to encode policy documents and customer queries.
- **Vector retrieval:** Uses FAISS to retrieve relevant policies by similarity.
- **Retrieval-Augmented Generation (RAG):** Supplies retrieved policy content to Gemini when generating answers.
- **Source-grounded responses:** Encourages answers to cite the relevant policy filenames.
- **Conversation context:** Passes recent chat history to the answer-generation function.
- **Interactive interface:** Provides a chat interface built with Gradio.
- **Retrieval tests:** Includes automated tests for policy loading, relevant-document retrieval, empty queries, and chatbot configuration.

## Architecture

```text
Customer question
       |
       v
Sentence Transformer
       |
       v
FAISS similarity search
       |
       v
Top relevant policy documents
       |
       v
Policy context + recent conversation
       |
       v
Gemini answer generation
       |
       v
Response with policy source citations
       |
       v
Gradio chat interface
```

### Technology Stack

| Component | Technology |
|---|---|
| Programming language | Python |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector search | FAISS |
| Data processing | pandas, NumPy |
| Language model | Gemini via `google-genai` |
| Chat interface | Gradio |
| Development environment | Google Colab |
| Version control | Git and GitHub |

## Repository Structure

```text
ShopAssist/
├── app/
│   └── gradio_app.py
├── data/
│   ├── policies/
│   └── evaluation/
├── notebooks/
│   ├── 01_data_exploration.ipynb
│   └── 02_chatbot_app.ipynb
├── src/
│   ├── retrieval.py
│   └── chatbot.py
├── tests/
│   └── test_retrieval.py
├── .gitignore
├── requirements.txt
└── README.md
```

## Knowledge Base

The knowledge base contains 15 Markdown policy documents covering:

- Account policies
- Cancellations
- Damaged and missing items
- Delivery and shipping
- Discounts and gift cards
- International shipping
- Payment methods and privacy
- Refunds, replacements, and returns
- Warranty information

The policies are stored in `data/policies/`.

## Retrieval Evaluation

The initial retrieval evaluation used a set of customer questions with expected relevant policy documents.

| Metric | Recorded result |
|---|---:|
| Mean Reciprocal Rank (MRR) | 0.9474 |
| Recall@3 | 100% |

These figures describe the earlier retrieval evaluation recorded during development. They are retrieval metrics, not measurements of overall answer correctness or customer satisfaction.

The project also includes `data/evaluation/rag_evaluation_results.csv` for reviewing previously recorded answer-generation evaluation results.

## Automated Tests

The retrieval test suite is located at `tests/test_retrieval.py`.

The five recorded retrieval tests passed during development. They check:

1. Loading all 15 policy documents.
2. Confirming policy sources use Markdown filenames.
3. Retrieving the damaged-item policy for a damaged-item query.
4. Retrieving the delivery policy for a standard-delivery query.
5. Returning no results for an empty query.

A separate configuration check also passed for handling a missing Gemini API key.

Run the retrieval tests from the project root:

```bash
python tests/test_retrieval.py
```

The first run may download the embedding model if it is not cached.

## Setup and Usage

### 1. Install dependencies

Use Python in an environment compatible with the packages listed in `requirements.txt`.

```bash
pip install -r requirements.txt
```

### 2. Configure the Gemini API key

Set the `GEMINI_API_KEY` environment variable securely. Never commit API keys or other credentials to GitHub.

For example, in a terminal:

```bash
export GEMINI_API_KEY="your-api-key"
```

Use the equivalent environment-variable configuration for your operating system or deployment environment.

### 3. Launch the application

From the repository root:

```bash
python app/gradio_app.py
```

The application loads the policy documents, builds the retrieval index, and launches the Gradio interface. A public share URL may be generated by Gradio; such links are temporary and are not a permanent deployment.

### Example Questions

- How long does standard delivery take?
- How can I return an item?
- My item arrived damaged. What should I do?
- Can I change my delivery address after dispatch?

## Limitations

- Responses depend on the completeness and accuracy of the available policy documents.
- Semantic retrieval can return relevant but not always optimal documents.
- Source citations are generated as part of the model response and are not independently verified against every claim.
- Answer quality has not been established through a comprehensive, independently validated evaluation.
- Gemini API availability and rate limits may affect response generation.
- The current Gradio share link is intended for temporary testing, not permanent production hosting.

## Future Improvements

- Expand and strengthen automated evaluation of retrieval and answer correctness.
- Improve handling of follow-up questions and ambiguous customer requests.
- Validate citations and claims against retrieved policy text.
- Add more robust error handling, logging, and monitoring.
- Deploy the application to a persistent hosting environment.
- Add broader unit and integration tests.

## Project Status

The modular retrieval and chatbot code has been created, the Gradio application has launched successfully in Google Colab, and initial retrieval and manual chatbot checks have been completed.

---

Developed as an AI customer-support portfolio project.
"""

readme_path.write_text(readme_content.strip() + "\n", encoding="utf-8")

print("README updated:", readme_path)
print("Characters:", len(readme_content))
print("\nPreview:\n")
print("\n".join(readme_path.read_text(encoding="utf-8").splitlines()[:25]))

README updated: /content/ShopAssist/README.md
Characters: 6420

Preview:

# ShopAssist AI — Intelligent E-commerce Customer Support Agent

ShopAssist AI is a retrieval-augmented customer support chatbot designed to answer common e-commerce questions using a collection of policy documents. It combines semantic search, vector retrieval, a large language model, and a Gradio chat interface to provide answers grounded in relevant policies.

## Project Overview

E-commerce customers often need quick, consistent answers about delivery, returns, refunds, replacements, payments, and other order-related questions.

ShopAssist retrieves relevant policy documents and uses them as context for generating responses. It is designed to cite its policy sources and acknowledge when the available documentation does not specify an answer.

## Key Features

- **Semantic search:** Uses Sentence Transformers to encode policy documents and customer queries.
- **Vector retrieval:** Uses FAISS to retrieve releva

In [60]:
import subprocess

result = subprocess.run(
    ["git", "diff", "--check"],
    cwd="/content/ShopAssist",
    capture_output=True,
    text=True
)
print(result.stdout or "No whitespace errors.")
print(result.stderr)

result = subprocess.run(
    ["git", "status", "--short"],
    cwd="/content/ShopAssist",
    capture_output=True,
    text=True
)
print("\nGit status:\n", result.stdout)

No whitespace errors.


Git status:
 


In [61]:
import subprocess
from google.colab import userdata

repo_path = "/content/ShopAssist"
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

def run_git(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=repo_path,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

# Stage and commit only the README
run_git(["add", "--", "README.md"])
run_git(["commit", "-m", "Improve project documentation"])

print("README commit created successfully.")

# Push without displaying the token
remote_url = (
    "https://x-access-token:"
    + token
    + "@github.com/DhruviPatadia/ShopAssist.git"
)

result = subprocess.run(
    ["git", "push", remote_url, "HEAD"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("README pushed successfully!")
else:
    print("Push failed:")
    print(result.stderr)

RuntimeError: 

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

for filename in [
    "evaluation_questions.csv",
    "rag_evaluation_results.csv"
]:
    path = EVALUATION_DIR / filename

    print(f"\n{'=' * 60}")
    print(filename)

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(5))
    else:
        print("File not found:", path)

In [62]:
from pathlib import Path
import pandas as pd
import re

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

questions_df = pd.read_csv(EVALUATION_DIR / "evaluation_questions.csv")
results_df = pd.read_csv(EVALUATION_DIR / "rag_evaluation_results.csv")

# 1. Source retrieval accuracy
results_df["source_correct"] = (
    results_df["expected_source"].str.strip()
    == results_df["retrieved_source"].str.strip()
)

source_accuracy = results_df["source_correct"].mean()

print(f"Source retrieval accuracy: {source_accuracy:.2%}")
print(
    f"Correct source: {results_df['source_correct'].sum()}"
    f"/{len(results_df)}"
)

# 2. Lightweight expected-answer keyword/phrase check
def answer_check(expected, generated):
    if pd.isna(expected) or pd.isna(generated):
        return False

    expected_text = str(expected).strip().lower()
    generated_text = str(generated).strip().lower()

    if not expected_text:
        return False

    # Direct phrase match
    if expected_text in generated_text:
        return True

    # For short expected answers, avoid unreliable substring checks
    # such as matching "yes" inside another word.
    expected_words = re.findall(r"\b[\w–-]+\b", expected_text)
    generated_words = set(re.findall(r"\b[\w–-]+\b", generated_text))

    if len(expected_words) <= 2:
        return all(word in generated_words for word in expected_words)

    return False

results_df["answer_keyword_check"] = results_df.apply(
    lambda row: answer_check(
        row["expected_answer"],
        row["generated_answer"]
    ),
    axis=1
)

keyword_rate = results_df["answer_keyword_check"].mean()

print(f"\nExpected-answer keyword/phrase check rate: {keyword_rate:.2%}")
print(
    f"Passed checks: {results_df['answer_keyword_check'].sum()}"
    f"/{len(results_df)}"
)

# 3. Query-type breakdown
if "query_type" in questions_df.columns:
    type_df = questions_df[["question", "query_type"]].merge(
        results_df[["question", "source_correct", "answer_keyword_check"]],
        on="question",
        how="left"
    )

    type_summary = type_df.groupby("query_type", dropna=False).agg(
        questions=("question", "count"),
        source_accuracy=("source_correct", "mean"),
        keyword_check_rate=("answer_keyword_check", "mean")
    ).reset_index()

    display(type_summary)

# 4. Inspect rows needing review
print("\nRows where the expected source was not retrieved:")
display(
    results_df.loc[
        ~results_df["source_correct"],
        ["question", "expected_source", "retrieved_source"]
    ]
)

print("\nRows failing the answer keyword/phrase check:")
display(
    results_df.loc[
        ~results_df["answer_keyword_check"],
        ["question", "expected_answer", "generated_answer"]
    ]
)

Source retrieval accuracy: 85.00%
Correct source: 17/20

Expected-answer keyword/phrase check rate: 40.00%
Passed checks: 8/20


,query_type,questions,source_accuracy,keyword_check_rate
0,customer_scenario,3,1.000000,0.0
1,direct,12,0.916667,0.5
2,out_of_scope,1,0.000000,0.0
3,paraphrased,4,0.750000,0.5



Rows where the expected source was not retrieved:


,question,expected_source,retrieved_source
8,Do you ship internationally?,shipping_policy.md,international_shipping.md
9,Can you deliver orders outside the country?,shipping_policy.md,international_shipping.md
19,Can I change my delivery address after the ord...,NaN,delivery_policy.md



Rows failing the answer keyword/phrase check:


,question,expected_answer,generated_answer
2,Can I return a damaged product?,Yes,Items that are damaged when received are handl...
3,My item arrived damaged. What should I do?,Contact support and provide photographs if req...,"If your item arrived damaged, you should conta..."
4,How long does standard delivery take?,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...
5,When should I expect my standard delivery?,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...
8,Do you ship internationally?,"Yes, to selected countries","Yes, we offer international shipping to select..."
9,Can you deliver orders outside the country?,"Yes, to selected countries","Yes, we offer international shipping to select..."
11,I received the wrong product. Can you send me ...,"Yes, a replacement may be requested","Yes, you may request a replacement if you rece..."
13,What payment methods do you accept?,Major debit and credit cards and other methods...,"Hello! We accept major debit and credit cards,..."
14,Can I use more than one discount code?,Normally only one promotional discount code,Only one promotional discount code can normall...
15,What happens if something is missing from my d...,Contact support and the issue will be investig...,If an expected item is missing from your deliv...


In [63]:
from pathlib import Path
import pandas as pd
import re

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

results_df = pd.read_csv(
    EVALUATION_DIR / "rag_evaluation_results.csv"
)

# Explicitly identify a known alternative source for international shipping.
# This mapping should be reviewed against the actual policy contents.
alternative_sources = {
    "shipping_policy.md": {"international_shipping.md"}
}

def exact_source_match(row):
    expected = row["expected_source"]
    retrieved = row["retrieved_source"]

    if pd.isna(expected) or pd.isna(retrieved):
        return None

    return str(expected).strip() == str(retrieved).strip()

def source_relevant(row):
    expected = row["expected_source"]
    retrieved = row["retrieved_source"]

    if pd.isna(expected) or pd.isna(retrieved):
        return None

    expected = str(expected).strip()
    retrieved = str(retrieved).strip()

    return (
        expected == retrieved
        or retrieved in alternative_sources.get(expected, set())
    )

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip()

results_df["exact_source_match"] = results_df.apply(
    exact_source_match, axis=1
)
results_df["source_relevant_under_mapping"] = results_df.apply(
    source_relevant, axis=1
)

# A conservative lexical diagnostic, not a semantic correctness metric.
results_df["expected_phrase_present"] = results_df.apply(
    lambda row: (
        normalize_text(row["expected_answer"]) in
        normalize_text(row["generated_answer"])
        if normalize_text(row["expected_answer"])
        else False
    ),
    axis=1
)

results_df["needs_manual_review"] = (
    results_df["source_relevant_under_mapping"].ne(True)
    | ~results_df["expected_phrase_present"]
)

output_path = EVALUATION_DIR / "improved_evaluation_report.csv"
results_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\nExact source match:")
print(
    f"{results_df['exact_source_match'].eq(True).sum()} / "
    f"{results_df['exact_source_match'].notna().sum()}"
)

valid_relevance = results_df["source_relevant_under_mapping"].dropna()
print("\nSource relevance under the explicit mapping:")
print(f"{valid_relevance.sum()} / {len(valid_relevance)}")

print("\nExpected phrase present:")
print(
    f"{results_df['expected_phrase_present'].sum()} / "
    f"{len(results_df)}"
)

print("\nRows requiring manual review:")
display(
    results_df.loc[
        results_df["needs_manual_review"],
        [
            "question",
            "expected_source",
            "retrieved_source",
            "expected_answer",
            "generated_answer"
        ]
    ]
)

Saved: /content/drive/MyDrive/ShopAssist/data/evaluation/improved_evaluation_report.csv

Exact source match:
17 / 19

Source relevance under the explicit mapping:
19 / 19

Expected phrase present:
8 / 20

Rows requiring manual review:


,question,expected_source,retrieved_source,expected_answer,generated_answer
2,Can I return a damaged product?,damaged_items.md,damaged_items.md,Yes,Items that are damaged when received are handl...
3,My item arrived damaged. What should I do?,damaged_items.md,damaged_items.md,Contact support and provide photographs if req...,"If your item arrived damaged, you should conta..."
4,How long does standard delivery take?,delivery_policy.md,delivery_policy.md,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...
5,When should I expect my standard delivery?,delivery_policy.md,delivery_policy.md,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...
8,Do you ship internationally?,shipping_policy.md,international_shipping.md,"Yes, to selected countries","Yes, we offer international shipping to select..."
9,Can you deliver orders outside the country?,shipping_policy.md,international_shipping.md,"Yes, to selected countries","Yes, we offer international shipping to select..."
11,I received the wrong product. Can you send me ...,replacement_policy.md,replacement_policy.md,"Yes, a replacement may be requested","Yes, you may request a replacement if you rece..."
13,What payment methods do you accept?,payment_policy.md,payment_policy.md,Major debit and credit cards and other methods...,"Hello! We accept major debit and credit cards,..."
14,Can I use more than one discount code?,discount_policy.md,discount_policy.md,Normally only one promotional discount code,Only one promotional discount code can normall...
15,What happens if something is missing from my d...,missing_items.md,missing_items.md,Contact support and the issue will be investig...,If an expected item is missing from your deliv...


In [64]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/ShopAssist")
EVALUATION_DIR = PROJECT_DIR / "data" / "evaluation"

results_df = pd.read_csv(
    EVALUATION_DIR / "rag_evaluation_results.csv"
)

review_df = results_df[
    [
        "question",
        "expected_source",
        "expected_answer",
        "generated_answer",
        "retrieved_source"
    ]
].copy()

review_df["human_verdict"] = ""
review_df["source_supported"] = ""
review_df["citation_correct"] = ""
review_df["review_notes"] = ""

review_path = EVALUATION_DIR / "human_review_benchmark.csv"
review_df.to_csv(review_path, index=False)

print("Created:", review_path)
print("Rows to review:", len(review_df))
print("\nFirst five rows:")
display(review_df.head())

Created: /content/drive/MyDrive/ShopAssist/data/evaluation/human_review_benchmark.csv
Rows to review: 20

First five rows:


,question,expected_source,expected_answer,generated_answer,retrieved_source,human_verdict,source_supported,citation_correct,review_notes
0,How long do I have to return an item?,returns_policy.md,30 days,You have 30 days from delivery to return an el...,returns_policy.md,,,,
1,What's the deadline for sending something back?,returns_policy.md,30 days,You may return eligible products within 30 day...,returns_policy.md,,,,
2,Can I return a damaged product?,damaged_items.md,Yes,Items that are damaged when received are handl...,damaged_items.md,,,,
3,My item arrived damaged. What should I do?,damaged_items.md,Contact support and provide photographs if req...,"If your item arrived damaged, you should conta...",damaged_items.md,,,,
4,How long does standard delivery take?,delivery_policy.md,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...,delivery_policy.md,,,,


In [65]:
from pathlib import Path
import pandas as pd
from IPython.display import display

file_path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/human_review_benchmark.csv"
)

df = pd.read_csv(file_path)

print("File:", file_path.name)
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

display(df.head())

File: human_review_benchmark.csv
Columns: ['question', 'expected_source', 'expected_answer', 'generated_answer', 'retrieved_source', 'human_verdict', 'source_supported', 'citation_correct', 'review_notes']
Shape: (20, 9)


,question,expected_source,expected_answer,generated_answer,retrieved_source,human_verdict,source_supported,citation_correct,review_notes
0,How long do I have to return an item?,returns_policy.md,30 days,You have 30 days from delivery to return an el...,returns_policy.md,NaN,NaN,NaN,NaN
1,What's the deadline for sending something back?,returns_policy.md,30 days,You may return eligible products within 30 day...,returns_policy.md,NaN,NaN,NaN,NaN
2,Can I return a damaged product?,damaged_items.md,Yes,Items that are damaged when received are handl...,damaged_items.md,NaN,NaN,NaN,NaN
3,My item arrived damaged. What should I do?,damaged_items.md,Contact support and provide photographs if req...,"If your item arrived damaged, you should conta...",damaged_items.md,NaN,NaN,NaN,NaN
4,How long does standard delivery take?,delivery_policy.md,3–5 working days after dispatch,Standard domestic delivery takes 3–5 working d...,delivery_policy.md,NaN,NaN,NaN,NaN


In [66]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/"
    "human_review_benchmark.csv"
)

df = pd.read_csv(path)

reviews = [
    ("correct", "yes", "yes", "Correct return deadline; verify against returns policy."),
    ("correct", "yes", "yes", "Correct deadline; additional replacement detail."),
    ("correct", "yes", "yes", "Damaged-item guidance and possible remedies."),
    ("correct", "yes", "yes", "Contact support and photo guidance."),
    ("correct", "yes", "yes", "Correct delivery estimate after dispatch."),
    ("correct", "yes", "yes", "Correct estimate; processing time is distinguished."),
    ("correct", "yes", "yes", "Cancellation normally unavailable after dispatch."),
    ("correct", "yes", "yes", "Consistent with the cancellation policy."),
    ("correct", "yes", "no", "Retrieved international_shipping.md; verify accepted source mapping."),
    ("correct", "yes", "yes", "International shipping answer; verify both cited policies."),
    ("correct", "yes", "yes", "Correct replacement guidance for wrong item."),
    ("correct", "yes", "yes", "Correct replacement guidance and refund alternative."),
    ("correct", "yes", "yes", "Correct refund estimate; verify policy wording."),
    ("correct", "yes", "yes", "Matches expected payment methods."),
    ("correct", "yes", "yes", "Correct discount-code rule and exception."),
    ("correct", "yes", "yes", "Appropriate investigation steps for missing items."),
    ("correct", "yes", "yes", "Matches warranty qualification."),
    ("correct", "yes", "yes", "Correct gift-card cash-exchange rule."),
    ("correct", "yes", "yes", "Correct privacy guidance."),
    ("correct", "yes", "not_applicable", "Appropriate uncertainty; verify cited policies lack an answer.")
]

assert len(df) == len(reviews) == 20, (
    f"Expected 20 rows, found {len(df)}"
)

for i, (verdict, supported, citation, notes) in enumerate(reviews):
    df.loc[i, "human_verdict"] = verdict
    df.loc[i, "source_supported"] = supported
    df.loc[i, "citation_correct"] = citation
    df.loc[i, "review_notes"] = notes

df.to_csv(path, index=False)

print("Saved preliminary review to:", path)
display(df[[
    "question",
    "human_verdict",
    "source_supported",
    "citation_correct",
    "review_notes"
]])

Saved preliminary review to: /content/drive/MyDrive/ShopAssist/data/evaluation/human_review_benchmark.csv


/tmp/ipykernel_8580/1673536630.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'correct' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[i, "human_verdict"] = verdict
/tmp/ipykernel_8580/1673536630.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[i, "source_supported"] = supported
/tmp/ipykernel_8580/1673536630.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[i, "citation_correct"] = citation
/tmp/ipykernel_8580/1673536630.py:42: FutureWarning: Setting an item of incompatible dt

,question,human_verdict,source_supported,citation_correct,review_notes
0,How long do I have to return an item?,correct,yes,yes,Correct return deadline; verify against return...
1,What's the deadline for sending something back?,correct,yes,yes,Correct deadline; additional replacement detail.
2,Can I return a damaged product?,correct,yes,yes,Damaged-item guidance and possible remedies.
3,My item arrived damaged. What should I do?,correct,yes,yes,Contact support and photo guidance.
4,How long does standard delivery take?,correct,yes,yes,Correct delivery estimate after dispatch.
5,When should I expect my standard delivery?,correct,yes,yes,Correct estimate; processing time is distingui...
6,Can I cancel my order after it has been dispat...,correct,yes,yes,Cancellation normally unavailable after dispatch.
7,Is it possible to cancel an order once it has ...,correct,yes,yes,Consistent with the cancellation policy.
8,Do you ship internationally?,correct,yes,no,Retrieved international_shipping.md; verify ac...
9,Can you deliver orders outside the country?,correct,yes,yes,International shipping answer; verify both cit...


In [67]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/"
    "human_review_benchmark.csv"
)

df = pd.read_csv(path)

review_columns = [
    "human_verdict",
    "source_supported",
    "citation_correct",
    "review_notes"
]

# Explicitly set review columns to string/object-compatible values
for column in review_columns:
    df[column] = df[column].fillna("").astype(str)

df.to_csv(path, index=False)

# Reload to verify the saved file
check = pd.read_csv(path, dtype=str).fillna("")

print("File saved and verified.")
print("Rows:", len(check))
print("\nReview verdict counts:")
print(check["human_verdict"].value_counts().to_string())
print("\nMissing review labels:")
print((check[review_columns[:3]] == "").sum().to_string())

File saved and verified.
Rows: 20

Review verdict counts:
human_verdict
correct    20

Missing review labels:
human_verdict       0
source_supported    0
citation_correct    0


In [68]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/"
    "human_review_benchmark.csv"
)

df = pd.read_csv(path).fillna("")

total = len(df)

print("SHOPASSIST AI — PRELIMINARY EVALUATION")
print("=" * 45)
print(f"Total questions: {total}")

print("\nHuman verdict distribution:")
print(df["human_verdict"].value_counts().to_string())

print("\nPreliminary metrics:")
print(
    "Answer correctness:",
    f"{(df['human_verdict'] == 'correct').sum() / total * 100:.1f}%"
)
print(
    "Source support:",
    f"{(df['source_supported'] == 'yes').sum() / total * 100:.1f}%"
)

applicable = df["citation_correct"].isin(["yes", "no"])
if applicable.sum():
    citation_accuracy = (
        (df.loc[applicable, "citation_correct"] == "yes").sum()
        / applicable.sum()
        * 100
    )
    print(
        "Citation accuracy among applicable labels:",
        f"{citation_accuracy:.1f}%"
    )
    print("Applicable citation labels:", int(applicable.sum()))

print("\nReminder: these are preliminary, not independently verified metrics.")

SHOPASSIST AI — PRELIMINARY EVALUATION
Total questions: 20

Human verdict distribution:
human_verdict
correct    20

Preliminary metrics:
Answer correctness: 100.0%
Source support: 100.0%
Citation accuracy among applicable labels: 94.7%
Applicable citation labels: 19

Reminder: these are preliminary, not independently verified metrics.


In [69]:
from pathlib import Path

policy_dir = Path(
    "/content/drive/MyDrive/ShopAssist/data/policies"
)

for filename in [
    "damaged_items.md",
    "shipping_policy.md",
    "international_shipping.md",
    "delivery_policy.md",
    "cancellation_policy.md",
]:
    path = policy_dir / filename
    print(f"\n{'=' * 60}")
    print(f"POLICY: {filename}")
    print("=" * 60)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("File not found.")


POLICY: damaged_items.md
# Damaged Items Policy

Customers should contact support as soon as possible if an item arrives damaged.

Customers may be asked to provide photographs of:
- the damaged product
- the packaging
- the shipping label

A damaged item may qualify for a replacement or refund depending on the circumstances.

Customers should not dispose of the damaged product or packaging until support has advised them to do so.

Damage caused after delivery through misuse or accidental damage may not qualify for a replacement or refund under this policy.

POLICY: shipping_policy.md
# Shipping Policy

We offer domestic shipping and international shipping to selected countries.

Available shipping methods and their costs are displayed during checkout.

Customers are responsible for providing an accurate delivery address.

Once an order has been dispatched, the customer may receive tracking information when tracking is available.

Shipping times may vary depending on the destination, 

In [70]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/"
    "human_review_benchmark.csv"
)

df = pd.read_csv(path, dtype=str).fillna("")

# Q9 is the ninth row (zero-based index 8)
df.loc[8, "citation_correct"] = "yes"
df.loc[8, "review_notes"] = (
    "Verified: international_shipping.md explicitly confirms "
    "shipping to selected countries."
)

df.to_csv(path, index=False)

print("Updated Q9 citation review.")
print(df.loc[8, [
    "question",
    "retrieved_source",
    "citation_correct",
    "review_notes"
]].to_string())

Updated Q9 citation review.
question                                 Do you ship internationally?
retrieved_source                            international_shipping.md
citation_correct                                                  yes
review_notes        Verified: international_shipping.md explicitly...


In [71]:
df = pd.read_csv(path, dtype=str).fillna("")

applicable = df["citation_correct"].isin(["yes", "no"])
correct_citations = (
    df.loc[applicable, "citation_correct"] == "yes"
).sum()

print("Applicable citations:", int(applicable.sum()))
print("Correct citations:", int(correct_citations))
print(
    "Citation accuracy:",
    f"{correct_citations / applicable.sum() * 100:.1f}%"
)

Applicable citations: 19
Correct citations: 19
Citation accuracy: 100.0%


In [72]:
from pathlib import Path

policy_dir = Path(
    "/content/drive/MyDrive/ShopAssist/data/policies"
)

files_to_check = [
    "returns_policy.md",
    "replacement_policy.md",
    "refund_policy.md",
    "payment_policy.md",
    "discount_policy.md",
    "missing_items.md",
    "warranty_policy.md",
    "gift_cards.md",
    "privacy_policy.md",
]

for filename in files_to_check:
    file_path = policy_dir / filename

    print(f"\n{'=' * 60}")
    print(f"POLICY: {filename}")
    print("=" * 60)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


POLICY: returns_policy.md
# Returns Policy

Customers may return eligible products within 30 days of delivery.

Products must be:
- unused
- undamaged
- in their original packaging
- accompanied by proof of purchase where required

Customers should initiate a return through the customer account or by contacting customer support.

Certain products may be non-returnable, including personalised products, hygiene-sensitive products that have been opened, and final-sale items.

Items that are damaged or incorrect when received are handled under the damaged-items and replacement policies.

POLICY: replacement_policy.md
# Replacement Policy

Customers may request a replacement if they receive:
- the wrong product
- a damaged product
- a defective product
- an item that is missing essential components

Replacement requests should normally be submitted within 30 days of delivery.

If the requested product is unavailable, the customer may be offered a refund instead.

Customers may be asked to 

In [73]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation/"
    "human_review_benchmark.csv"
)

df = pd.read_csv(path, dtype=str).fillna("")

# Q20 is the final row
df.loc[19, "source_supported"] = "uncertain"
df.loc[19, "review_notes"] = (
    "The chatbot appropriately acknowledges missing information. "
    "The available policies do not specify whether an address can "
    "be changed after dispatch, so source support is uncertain."
)

df.to_csv(path, index=False)

print("Updated Q20.")
print(df.loc[19, [
    "question",
    "human_verdict",
    "source_supported",
    "citation_correct",
    "review_notes"
]].to_string(index=False))

Updated Q20.
Can I change my delivery address after the orde...
                                           correct
                                         uncertain
                                    not_applicable
The chatbot appropriately acknowledges missing ...


In [74]:
df = pd.read_csv(path, dtype=str).fillna("")

total = len(df)
correct_answers = (df["human_verdict"] == "correct").sum()

source_labels = df["source_supported"].str.lower()
source_yes = (source_labels == "yes").sum()
source_no = (source_labels == "no").sum()
source_uncertain = (source_labels == "uncertain").sum()

citation_labels = df["citation_correct"].str.lower()
citation_applicable = citation_labels.isin(["yes", "no"])
citation_correct = (citation_labels == "yes").sum()

print("SHOPASSIST AI — POLICY-REVIEWED EVALUATION")
print("=" * 48)
print(f"Questions reviewed: {total}")
print(f"Answers labelled correct: {correct_answers}/{total}")
print(f"Answer correctness: {correct_answers / total * 100:.1f}%")

print(f"\nSource supported: {source_yes}/{total}")
print(f"Source unsupported: {source_no}/{total}")
print(f"Source support uncertain: {source_uncertain}/{total}")

if citation_applicable.sum():
    print(
        "\nCitation accuracy:",
        f"{citation_correct / citation_applicable.sum() * 100:.1f}%"
    )
    print("Applicable citations:", int(citation_applicable.sum()))

SHOPASSIST AI — POLICY-REVIEWED EVALUATION
Questions reviewed: 20
Answers labelled correct: 20/20
Answer correctness: 100.0%

Source supported: 19/20
Source unsupported: 0/20
Source support uncertain: 1/20

Citation accuracy: 100.0%
Applicable citations: 19


In [75]:
import pandas as pd
from pathlib import Path

project_root = Path("/content/drive/MyDrive/ShopAssist")
evaluation_dir = project_root / "data" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

benchmark_path = evaluation_dir / "human_review_benchmark.csv"
df = pd.read_csv(benchmark_path, dtype=str).fillna("")

total = len(df)
correct = (df["human_verdict"] == "correct").sum()
partial = (df["human_verdict"] == "partially_correct").sum()
incorrect = (df["human_verdict"] == "incorrect").sum()

source_yes = (df["source_supported"] == "yes").sum()
source_no = (df["source_supported"] == "no").sum()
source_uncertain = (df["source_supported"] == "uncertain").sum()

citation_applicable = df["citation_correct"].isin(["yes", "no"])
citation_correct = (df.loc[citation_applicable, "citation_correct"] == "yes").sum()
citation_total = int(citation_applicable.sum())

metrics = pd.DataFrame([
    ["Questions reviewed", total, "Count"],
    ["Labelled correct", int(correct), "Count"],
    ["Preliminary answer correctness", round(correct / total * 100, 1), "%"],
    ["Source supported", int(source_yes), "Count"],
    ["Source unsupported", int(source_no), "Count"],
    ["Source uncertain", int(source_uncertain), "Count"],
    ["Citation correct", int(citation_correct), "Count"],
    ["Applicable citations", citation_total, "Count"],
    [
        "Citation accuracy",
        round(citation_correct / citation_total * 100, 1)
        if citation_total else 0,
        "%"
    ],
], columns=["metric", "value", "unit"])

output_path = evaluation_dir / "human_review_metrics.csv"
metrics.to_csv(output_path, index=False)

print("Saved metrics report:", output_path)
display(metrics)

Saved metrics report: /content/drive/MyDrive/ShopAssist/data/evaluation/human_review_metrics.csv


,metric,value,unit
0,Questions reviewed,20.0,Count
1,Labelled correct,20.0,Count
2,Preliminary answer correctness,100.0,%
3,Source supported,19.0,Count
4,Source unsupported,0.0,Count
5,Source uncertain,1.0,Count
6,Citation correct,19.0,Count
7,Applicable citations,19.0,Count
8,Citation accuracy,100.0,%


In [77]:
from pathlib import Path

readme_path = Path("/content/ShopAssist/README.md")
readme = readme_path.read_text(encoding="utf-8")

section_heading = "## Human-Reviewed Answer Evaluation"

new_section = """## Human-Reviewed Answer Evaluation

ShopAssist AI was evaluated using a benchmark of 20 customer questions covering returns, delivery, cancellations, international shipping, replacements, refunds, payments, discounts, missing items, warranties, gift cards, privacy, and an out-of-scope address-change question.

The benchmark compares generated responses with expected answers and relevant policy documents. Responses were reviewed using three criteria: answer correctness, policy-source support, and citation correctness.

### Preliminary Results

| Metric | Result |
|---|---:|
| Questions reviewed | 20 |
| Answers labelled correct | 20/20 (100%) |
| Answers marked as source-supported | 19/20 (95%) |
| Answers with uncertain source support | 1/20 (5%) |
| Citation accuracy among applicable labels | 19/19 (100%) |

**Interpretation:** These are preliminary, human-reviewed results based on the current benchmark labels. They should not be interpreted as independently validated model accuracy or as evidence of performance across all possible customer queries.

One question asked whether a delivery address could be changed after dispatch. The available policies did not specify whether this was possible, and the chatbot acknowledged the information gap rather than asserting an unsupported policy.

Citation accuracy excludes that question because its citation label was marked `not_applicable`. The international-shipping source was also reviewed against the policy documents, which confirm shipping to selected countries.

### Evaluation Files

The evaluation artifacts are stored in `data/evaluation/`:

- `evaluation_questions.csv` — benchmark questions, expected answers, and expected sources.
- `rag_evaluation_results.csv` — recorded retrieval and generated-answer results.
- `improved_evaluation_report.csv` — additional retrieval and phrase-check indicators.
- `human_review_benchmark.csv` — human-review labels and notes.
- `human_review_metrics.csv` — summary of the preliminary human-review metrics.

The benchmark is small and reflects a limited set of policy questions. Further evaluation should include more paraphrases, ambiguous queries, unsupported questions, and independent review of answer claims and citations.

"""

if section_heading in readme:
    print("The evaluation section already exists. No changes made.")
elif "## Automated Tests" in readme:
    readme = readme.replace(
        "## Automated Tests",
        new_section + "## Automated Tests",
        1
    )
    readme_path.write_text(readme, encoding="utf-8")
    print("README updated successfully.")
else:
    print("Could not find the insertion point. README was not changed.")

print("\nREADME path:", readme_path)
print("Evaluation section present:", section_heading in readme_path.read_text(encoding="utf-8"))

README updated successfully.

README path: /content/ShopAssist/README.md
Evaluation section present: True


In [78]:
from pathlib import Path
import shutil

drive_eval = Path(
    "/content/drive/MyDrive/ShopAssist/data/evaluation"
)
repo_eval = Path(
    "/content/ShopAssist/data/evaluation"
)

repo_eval.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "improved_evaluation_report.csv",
    "human_review_benchmark.csv",
    "human_review_metrics.csv",
]

for filename in files_to_copy:
    source = drive_eval / filename
    destination = repo_eval / filename

    if source.exists():
        shutil.copy2(source, destination)
        print(f"Copied: {filename}")
    else:
        print(f"NOT FOUND: {source}")

print("\nEvaluation files now in repository:")
for file in sorted(repo_eval.glob("*.csv")):
    print(f"- {file.name}")

Copied: improved_evaluation_report.csv
Copied: human_review_benchmark.csv
Copied: human_review_metrics.csv

Evaluation files now in repository:
- evaluation_questions.csv
- human_review_benchmark.csv
- human_review_metrics.csv
- improved_evaluation_report.csv


In [79]:
import subprocess

result = subprocess.run(
    ["git", "status", "--short"],
    cwd="/content/ShopAssist",
    capture_output=True,
    text=True
)

print(result.stdout)

 M README.md
?? data/evaluation/human_review_benchmark.csv
?? data/evaluation/human_review_metrics.csv
?? data/evaluation/improved_evaluation_report.csv



In [80]:
import subprocess

repo = "/content/ShopAssist"

files = [
    "README.md",
    "data/evaluation/human_review_benchmark.csv",
    "data/evaluation/human_review_metrics.csv",
    "data/evaluation/improved_evaluation_report.csv",
]

# Stage only the intended files
subprocess.run(
    ["git", "add", *files],
    cwd=repo,
    check=True
)

# Review staged changes
result = subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)

print("Staged changes:")
print(result.stdout)

print("\nStaged file list:")
result = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout)

Staged changes:
 README.md                                      | 34 ++++++++++++++
 data/evaluation/human_review_benchmark.csv     | 65 ++++++++++++++++++++++++++
 data/evaluation/human_review_metrics.csv       | 10 ++++
 data/evaluation/improved_evaluation_report.csv | 65 ++++++++++++++++++++++++++
 4 files changed, 174 insertions(+)


Staged file list:
README.md
data/evaluation/human_review_benchmark.csv
data/evaluation/human_review_metrics.csv
data/evaluation/improved_evaluation_report.csv



In [81]:
import subprocess
from google.colab import userdata

repo = "/content/ShopAssist"

# Commit the staged files
commit = subprocess.run(
    [
        "git", "commit", "-m",
        "Add human-reviewed evaluation results"
    ],
    cwd=repo,
    capture_output=True,
    text=True
)

print(commit.stdout)

if commit.returncode != 0:
    print(commit.stderr)
    raise RuntimeError("Commit failed; inspect the output above.")

# Retrieve the GitHub token securely
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN was not available in Colab Secrets.")

# Push without changing the saved remote URL
push = subprocess.run(
    [
        "git", "push",
        "https://x-access-token:" + token
        + "@github.com/DhruviPatadia/ShopAssist.git",
        "HEAD"
    ],
    cwd=repo,
    capture_output=True,
    text=True
)

print("\nPush output:")
print(push.stdout)

if push.returncode != 0:
    print(push.stderr)
    raise RuntimeError("Push failed; inspect the output above.")

print("Successfully pushed the evaluation updates to GitHub.")

[main daba9b6] Add human-reviewed evaluation results
 4 files changed, 174 insertions(+)
 create mode 100644 data/evaluation/human_review_benchmark.csv
 create mode 100644 data/evaluation/human_review_metrics.csv
 create mode 100644 data/evaluation/improved_evaluation_report.csv


Push output:

Successfully pushed the evaluation updates to GitHub.
